In [49]:
print("Hello World!")

Hello World!


In [50]:
import requests
import pandas as pd
import defs

In [51]:
session = requests.Session()

In [52]:
ins_df = pd.read_pickle("instruments.pkl")

In [53]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD']

In [54]:
# ins_df.shape # num rows, num cols
# ins_df.info()
# ins_df.columns
# ins_df.name.unique()

In [55]:
def fetch_candles(pair_name, count, granularity):
    url = f"{defs.OANDA_URL}/instruments/{pair_name}/candles"
    params = dict(
        count = count,
        granularity = granularity,
        price = "MBA"
    )
    response = session.get(url, params=params, headers=defs.SECURE_HEADER)
    return response.status_code, response.json()


In [56]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [57]:
print(code)

200


In [58]:
print(res)

{'instrument': 'EUR_USD', 'granularity': 'H1', 'candles': [{'complete': True, 'volume': 5564, 'time': '2026-05-01T11:00:00.000000000Z', 'bid': {'o': '1.17441', 'h': '1.17575', 'l': '1.17432', 'c': '1.17506'}, 'mid': {'o': '1.17448', 'h': '1.17583', 'l': '1.17440', 'c': '1.17514'}, 'ask': {'o': '1.17456', 'h': '1.17591', 'l': '1.17448', 'c': '1.17522'}}, {'complete': True, 'volume': 9648, 'time': '2026-05-01T12:00:00.000000000Z', 'bid': {'o': '1.17505', 'h': '1.17680', 'l': '1.17475', 'c': '1.17567'}, 'mid': {'o': '1.17514', 'h': '1.17688', 'l': '1.17482', 'c': '1.17576'}, 'ask': {'o': '1.17523', 'h': '1.17697', 'l': '1.17490', 'c': '1.17584'}}, {'complete': True, 'volume': 11946, 'time': '2026-05-01T13:00:00.000000000Z', 'bid': {'o': '1.17567', 'h': '1.17630', 'l': '1.17432', 'c': '1.17609'}, 'mid': {'o': '1.17575', 'h': '1.17638', 'l': '1.17440', 'c': '1.17622'}, 'ask': {'o': '1.17583', 'h': '1.17647', 'l': '1.17448', 'c': '1.17634'}}, {'complete': True, 'volume': 15740, 'time': '2026

In [59]:
def get_candles_df(json_response):

    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']
    
    our_data = []
    for candle in json_response['candles']:
        if candle['complete'] == False:
            continue
        new_dict = {}
        new_dict['time'] = candle['time']
        new_dict['volume'] = candle['volume']
        for price in prices:
            for oh in ohlc:
                new_dict[f"{price}_{oh}"] = candle[price][oh]
        our_data.append(new_dict)
    return pd.DataFrame.from_dict(our_data)

In [60]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [61]:
df = get_candles_df(res)

In [62]:
df.head()

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2026-05-01T11:00:00.000000000Z,5564,1.17448,1.17583,1.17440,1.17514,1.17441,1.17575,1.17432,1.17506,1.17456,1.17591,1.17448,1.17522
1,2026-05-01T12:00:00.000000000Z,9648,1.17514,1.17688,1.17482,1.17576,1.17505,1.17680,1.17475,1.17567,1.17523,1.17697,1.17490,1.17584
2,2026-05-01T13:00:00.000000000Z,11946,1.17575,1.17638,1.17440,1.17622,1.17567,1.17630,1.17432,1.17609,1.17583,1.17647,1.17448,1.17634
3,2026-05-01T14:00:00.000000000Z,15740,1.17620,1.17852,1.17608,1.17674,1.17609,1.17845,1.17596,1.17667,1.17632,1.17861,1.17620,1.17682
4,2026-05-01T15:00:00.000000000Z,11728,1.17675,1.17708,1.17532,1.17550,1.17667,1.17701,1.17523,1.17543,1.17683,1.17716,1.17542,1.17558


In [63]:
def save_file(candles_df, pair, granularity):
    candles_df.to_pickle(f"his_data/{pair}_{granularity}.pkl")

In [64]:
def create_data(pair, granularity):
    code, json_data = fetch_candles(pair, 4000, granularity)
    if code != 200:
        print(pair, "Error")
        return
    df = get_candles_df(json_data)
    print(f"{pair} loaded {df.shape[0]} candles from {df.time.min()} to {df.time.max()}")
    save_file(df, pair, granularity)


In [65]:
for p1 in our_curr:
    for p2 in our_curr:
        pair = f"{p1}_{p2}"
        if pair in ins_df.name.unique():
            create_data(pair, "H1")

EUR_USD loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
EUR_GBP loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
EUR_JPY loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
EUR_CHF loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
EUR_NZD loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
EUR_CAD loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
USD_JPY loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
USD_CHF loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
USD_CAD loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
GBP_USD loaded 4000 candles from 2025-09-09T05:00:00.000000000Z to 2026-05-01T20:00:00.000000000Z
GBP_JPY loaded 4000 